# Step 2 — Feature Engineering & Preprocessing

Following exploratory analysis in `01_load_and_explore_data.ipynb`, this notebook maps those decisions to columns in `app/ml/features.py`.

Narrative length uses **`issueDesc_en`** when present (fallbacks in code); **money**, **EDA-style ratios**, **policy-span tier**, **symptoms**, and **persona**/`productName` are all represented below.


In [1]:
from pathlib import Path
import sys
import pandas as pd

# --- excerpt: app/ml/dataset.py — load_claims_training_frame ---
# y = (status.strip().lower() == "completed").astype(int); X = df.drop(columns=["status"])

_CWD = Path.cwd().resolve()
PROJECT_ROOT = _CWD.parent if _CWD.name == "notebooks" else _CWD
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.ml.dataset import load_claims_training_frame
from app.ml.features import (
    engineer_features,
    build_preprocessor,
    engineered_column_order,
    symptom_flag_columns,
)

data_path = PROJECT_ROOT / "claim_use_case_dataset.xlsx"
X, y = load_claims_training_frame(data_path)
print(f"Loaded {len(X)} rows. Target 'approved' rate: {float(y.mean()):.3f}")


Loaded 2880 rows. Target 'approved' rate: 0.843


In [9]:
ENG_SUBSET_N = 5000

X_engineered = engineer_features(X.iloc[:ENG_SUBSET_N].copy())

In [10]:
X_engineered

,excessFee,rrp,balanceRRP,oldBalanceRRP,productName,coverage,productCode,policyStatus,retailerName,deviceType,...,excessFee_log1p,rrp_log1p,balanceRRP_log1p,oldBalanceRRP_log1p,deviceCost_log1p,rrp_minus_balance,fee_to_rrp,symptom_count,persona,coverage_duration_tier
0,1989.0,11990.0,11990.0,11990.0,SE_ADLD+THEFT_12M_MONTHLY_SMARTPHONE,ADLD/THEFT,SEADLDTHEFT12,Active,WUAWEI eStore,NaN,...,7.595890,9.391912,9.391912,9.391912,0.0,0.0,0.165888,0.0,The Victim (Theft/Loss),tier_appx_12mo
1,619.0,15490.0,15490.0,15490.0,SE_MANDATORY_ADLD_12M_UPFRONT_SMARTPHONE,ADLD,SEADLD1206,Active,SWEDEN ESTORE BULK UPLOAD,NaN,...,6.429719,9.648014,9.648014,9.648014,0.0,0.0,0.039961,6.0,The Clumsy Dropper (Standard Accidental),tier_appx_12mo
2,2509.0,19490.0,19490.0,19490.0,SE_ADLD+THEFT_12M_MONTHLY_SMARTPHONE,ADLD/THEFT,SEADLDTHEFT12,Active,WUAWEI eStore,NaN,...,7.828038,9.877708,9.877708,9.877708,0.0,0.0,0.128733,0.0,The Victim (Theft/Loss),tier_appx_12mo
3,619.0,15490.0,15490.0,15490.0,SE_MANDATORY_ADLD_12M_UPFRONT_SMARTPHONE,ADLD,SEADLD1206,Active,SWEDEN ESTORE BULK UPLOAD,NaN,...,6.429719,9.648014,9.648014,9.648014,0.0,0.0,0.039961,6.0,The Clumsy Dropper (Standard Accidental),tier_appx_12mo
4,619.0,14490.0,14490.0,14490.0,SE_ADLD_24M_UPFRONT_SMARTPHONE,ADLD,SEADLD24,Active,WUAWEI eStore,NaN,...,6.429719,9.581283,9.581283,9.581283,0.0,0.0,0.042719,0.0,The Clumsy Dropper (Standard Accidental),tier_appx_24mo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2875,59.0,1009.0,1009.0,1009.0,FI_MANDATORY_ADLD_12M_MONTHLY_SMARTPHONE_12MTH...,ADLD,FIADLD1233,Active,NaN,SMARTPHONES,...,4.094345,6.917706,6.917706,6.917706,0.0,0.0,0.058474,6.0,The Clumsy Dropper (Standard Accidental),tier_appx_12mo
2876,59.0,1499.0,1499.0,1499.0,FI_MANDATORY_ADLD_12M_MONTHLY_SMARTPHONE_12MTH...,ADLD,FIADLD1233,Active,NaN,SMARTPHONES,...,4.094345,7.313220,7.313220,7.313220,0.0,0.0,0.039360,2.0,The Clumsy Dropper (Standard Accidental),tier_appx_12mo
2877,59.0,1649.0,1649.0,1649.0,FI_VOLUNTARY_ADLD_12M_MONTHLY_SMARTPHONE_UPSELL,ADLD,FIADLD1221,Active,NaN,SMARTPHONES,...,4.094345,7.408531,7.408531,7.408531,0.0,0.0,0.035779,2.0,The Clumsy Dropper (Standard Accidental),tier_appx_12mo
2878,179.0,3099.0,3099.0,3099.0,FI_VOLUNTARY_ADLD_12M_MONTHLY_LAPTOP_UPSELL,ADLD,FIADLD1224,Active,NaN,LAPTOP,...,5.192957,8.039157,8.039157,8.039157,0.0,0.0,0.057761,5.0,The Clumsy Dropper (Standard Accidental),tier_appx_12mo


## 1. Text length features (English narrative when available)
**EDA (Notebook 01):** After `issueDesc` is translated to `issueDesc_en`, we bucket **word counts** and plot approval rate (new section in 01). That motivates a compact numeric proxy — **character length** on the same English field — without embedding the full text.

**Logic (`engineer_features`):** Prefer `issueDesc_en`, then `issue_desc_en`, else `issueDesc`; strip empty/NaN placeholders only. `productName` / `productDesc` use the same light cleaning for length and categoricals. Raw narrative columns are dropped after lengths are computed; the model never sees free text, only `issue_desc_len`, `product_name_len`, `product_desc_len`.


In [11]:
# excerpt: app/ml/features.py — issue_desc_len
# _ISSUE_EN_COLS = ("issueDesc_en", "issue_desc_en", "issueDesc")
# issue_src = first matching column; desc_clean = issue_src.map(_simple_str_cell)
# out["issue_desc_len"] = desc_clean.map(lambda t: float(len(t)) if isinstance(t, str) else 0.0)
# drop_text: issueDesc, issueDesc_en, issue_desc_en, productDesc

print("Length features:")
display(X_engineered[["issue_desc_len", "product_name_len", "product_desc_len"]].head())


Length features:


,issue_desc_len,product_name_len,product_desc_len
0,549.0,36.0,54.0
1,205.0,40.0,44.0
2,572.0,36.0,54.0
3,722.0,40.0,44.0
4,192.0,30.0,42.0


## 2. Money signals (raw + ratios) and policy-span tier

**Clarifying Notebook 01:** **`excessFee` and `rrp` are predictive** chiefly via **chi-square / binned** views in Notebook 01 (quantile bins for list price, fee tiers)—not necessarily via large **Pearson $|r|$** versus a binary label. Small marginal correlations say little about usefulness for tree ensembles; splits can track non-monotone thresholds on scaled amounts.

**Numerics fed to median imputer + `StandardScaler`:** `excessFee`, `rrp`, `balanceRRP`, `oldBalanceRRP`, `deviceCost`, each with a parallel **`_log1p`**, plus **`rrp_minus_balance`** and **`fee_to_rrp`** (same spirit as Notebook 01; `fee_to_rrp` is **NaN** when `rrp` ≤ 0).

**Categorical (`OneHotEncoder`): `coverage_duration_tier`** — derived from `policy_length_days` with the same day cutoffs as Notebook 01 (including the `<120` day outlier bucket before the 6 m / 12 m / 24 m-style bands).


In [5]:
# excerpt: app/ml/features.py (after base money + log1p loop)
# rr = out["rrp"]; bal = out["balanceRRP"]; xf = out["excessFee"]
# out["rrp_minus_balance"] = rr - bal
# out["fee_to_rrp"] = np.where(rr > 0, xf / rr, np.nan)
# out["coverage_duration_tier"] = out["policy_length_days"].map(_coverage_duration_tier_from_days)

money_cols = [
    c
    for c in X_engineered.columns
    if any(
        k in c.lower()
        for k in ("rrp", "fee", "balance", "devicecost", "log1p", "minus", "fee_to_rrp")
    )
]
display(X_engineered[sorted(money_cols)].head())
print("coverage_duration_tier (categorical, OHE in preprocessor):")
print(X_engineered["coverage_duration_tier"].value_counts(dropna=False).head(10))


,balanceRRP,balanceRRP_log1p,deviceCost,deviceCost_log1p,excessFee,excessFee_log1p,fee_to_rrp,oldBalanceRRP,oldBalanceRRP_log1p,rrp,rrp_log1p,rrp_minus_balance
0,11990.0,9.391912,0.0,0.0,1989.0,7.595890,0.165888,11990.0,9.391912,11990.0,9.391912,0.0
1,15490.0,9.648014,0.0,0.0,619.0,6.429719,0.039961,15490.0,9.648014,15490.0,9.648014,0.0
2,19490.0,9.877708,0.0,0.0,2509.0,7.828038,0.128733,19490.0,9.877708,19490.0,9.877708,0.0
3,15490.0,9.648014,0.0,0.0,619.0,6.429719,0.039961,15490.0,9.648014,15490.0,9.648014,0.0
4,14490.0,9.581283,0.0,0.0,619.0,6.429719,0.042719,14490.0,9.581283,14490.0,9.581283,0.0


coverage_duration_tier (categorical, OHE in preprocessor):
coverage_duration_tier
tier_appx_12mo    299
tier_appx_24mo    146
tier_appx_6mo      42
outlier_lt120d     13
Name: count, dtype: int64


## 3. Symptom flags and `symptom_count`
**EDA (Notebook 01):** Pairwise combos and multi-symptom combinations in the Plotly scans (pairs through 12-plets, matching the twelve binary flags) show that **which** symptoms co-occur shifts approval — not only how many boxes are checked.

**Modeling trade-off:** Excel already stores each flag as numeric 0/1. Those **12 columns stay in the numeric matrix** (`StandardScaler` + median imputer) so specific combos remain expressible; `symptom_count` adds an explicit aggregation (sum). We do **not** one-hot symptom flags — `OneHotEncoder` applies only to categoricals (`model`, `claimType`, etc.). Raw 0/1 is not redundant with OHE: the latter is category expansion for strings, these are ordinal-friendly numerics entering the numeric pipeline unchanged apart from scaling.


In [6]:
# excerpt: app/ml/features.py
# _DAMAGE_FLOAT = ["turnOnOff", "touchScreen", "smashed", "frontCamera", "backCamera", "frontOrBackCamera", "audio", "mic", "buttons", "connection", "charging", "other"]
# out["symptom_count"] = out[_DAMAGE_FLOAT].fillna(0).sum(axis=1)
# numeric_engineered_names() includes base_nums + logs + symptom_count + _DAMAGE_FLOAT

damage_flags = list(symptom_flag_columns())
print(f"symptom_flag_columns(): {len(damage_flags)} flags")
display(X_engineered[damage_flags + ["symptom_count"]].head())


symptom_flag_columns(): 12 flags


,turnOnOff,touchScreen,smashed,frontCamera,backCamera,frontOrBackCamera,audio,mic,buttons,connection,charging,other,symptom_count
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,1.0,1.0,NaN,1.0,1.0,NaN,1.0,1.0,0.0,NaN,NaN,NaN,6.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,1.0,1.0,NaN,1.0,1.0,NaN,1.0,1.0,0.0,NaN,NaN,NaN,6.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


## 4. Rule-based persona fallback
**EDA Finding:** Persona clusters from incident text line up with distinct approval patterns.
**Logic:** If `persona` is missing, `engineer_features` applies regex rules to `claimType` and **`issueDesc_en` / `issue_desc_en` / `issueDesc`** (in that order).


In [7]:
# excerpt: app/ml/features.py — _assign_persona (abridged)
# if "theft" in claimType.lower() or regex(stole|stolen|...) -> "The Victim (Theft/Loss)"
# elif "liquid" in claimType or water|spilled|... -> "The Unlucky Spiller (Liquid Damage)"
# ... gym|sport -> Active; kid|pet -> Family; car|commute -> Commuter; work|office -> Professional
# else -> "The Clumsy Dropper (Standard Accidental)"

print("Persona distribution (engineered subset):")
print(X_engineered["persona"].value_counts())


Persona distribution (engineered subset):
persona
The Clumsy Dropper (Standard Accidental)    470
The Victim (Theft/Loss)                      20
The Unlucky Spiller (Liquid Damage)          10
Name: count, dtype: int64


## 5. Scikit-learn `ColumnTransformer`
**EDA Finding:** Nominal fields like `model` have very high cardinality.
**Logic:** `build_preprocessor()` — numeric pipeline: median impute + `StandardScaler`. Categorical pipeline: constant fill + `OneHotEncoder(max_categories=50, handle_unknown='infrequent_if_exist')`.


In [8]:
# excerpt: app/ml/features.py — build_preprocessor
# ColumnTransformer([("num", numeric_pipe, nums), ("cat", categorical_pipe, cats)], remainder="drop")

prep = build_preprocessor()
cols = engineered_column_order()
Xt = prep.fit_transform(X_engineered[cols])

print(f"Engineered frame (ordered columns): {X_engineered[cols].shape}")
print(f"Transformed matrix: {Xt.shape}")


Engineered frame (ordered columns): (500, 42)
Transformed matrix: (500, 143)


c:\coding\boltech\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['smashed' 'frontOrBackCamera' 'other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
